# 05 - Linear mixed models for phase effects

**Purpose**: per-feature test of whether kinematic measurements differ
across phases, accounting for:
- within-subject correlation (subject random intercept)
- within-session correlation (nested session random intercept)
- reach-level noise (residual)

**Input**: ``data.AKDdf`` (raw reaches) restricted to contacted reaches
and the analyzable phase set.

**Output**: three FDR-adjusted results tables, one per analysis:
- Omnibus 4-phase test
- Deficit (Baseline -> Post_Injury_2-4)
- Recovery (Post_Injury_2-4 -> Post_Rehab_Test)

**Small-sample caveat**: statsmodels uses a chi-square Wald
approximation, not Satterthwaite/Kenward-Roger. At small N subjects this
is mildly anti-conservative. Results here are suggestive of direction
and structure; revisit when N grows.


In [ ]:
# parameters
FEATURE_LIST = None   # None -> use helpers.kinematics.get_kinematic_cols(AKDdf) (deduped list)
TOP_N_FIGURE = 20     # How many features to show on the -log10(p) bar chart
FIGSIZE_BARS = (14, 16)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from endpoint_ck_analysis.config import ANALYZABLE_PHASES, EXAMPLE_OUTPUT_DIR, CACHE_DIR, FDR_ALPHA
from endpoint_ck_analysis.data_loader import load_all
from endpoint_ck_analysis.helpers.kinematics import get_kinematic_cols
from endpoint_ck_analysis.helpers.models import run_phase_lmm_for_features
from endpoint_ck_analysis.helpers.plotting import stamp_version

EXAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
data = load_all()


## 1. Prepare the reach-level dataframe

One row per contacted reach. Phase is an ordered Categorical with
Baseline as the reference level, so every fitted coefficient reads as
"phase X - Baseline".


In [ ]:
contacted = data.AKDdf[data.AKDdf['contact_group'] == 'contacted'].copy()
contacted['phase_group'] = pd.Categorical(
    contacted['phase_group'],
    categories=list(ANALYZABLE_PHASES),
    ordered=True,
)

features = FEATURE_LIST or get_kinematic_cols(contacted)
print(f'Features to test: {len(features)}')


## 2. Omnibus across all four phases


In [ ]:
omnibus = run_phase_lmm_for_features(contacted, features, fdr_alpha=FDR_ALPHA)
print(omnibus.head(15)[['feature', 'phase_p', 'phase_p_adj', 'n_reaches', 'n_subjects', 'converged']])
omnibus.to_parquet(CACHE_DIR / 'lmm_omnibus.parquet', index=False)


## 3. Deficit delta (Baseline vs Post_Injury_2-4)

Same model structure but restricted to two phases, so the phase
coefficient IS the delta.


In [ ]:
deficit_df = contacted[contacted['phase_group'].isin(['Baseline', 'Post_Injury_2-4'])].copy()
deficit_df['phase_group'] = pd.Categorical(deficit_df['phase_group'], categories=['Baseline', 'Post_Injury_2-4'])
deficit = run_phase_lmm_for_features(deficit_df, features, fdr_alpha=FDR_ALPHA)
print(deficit.head(15)[['feature', 'phase_p', 'phase_p_adj', 'n_reaches', 'n_subjects', 'converged']])
deficit.to_parquet(CACHE_DIR / 'lmm_deficit.parquet', index=False)


## 4. Recovery delta (Post_Injury_2-4 vs Post_Rehab_Test)


In [ ]:
recovery_df = contacted[contacted['phase_group'].isin(['Post_Injury_2-4', 'Post_Rehab_Test'])].copy()
recovery_df['phase_group'] = pd.Categorical(recovery_df['phase_group'], categories=['Post_Injury_2-4', 'Post_Rehab_Test'])
recovery = run_phase_lmm_for_features(recovery_df, features, fdr_alpha=FDR_ALPHA)
print(recovery.head(15)[['feature', 'phase_p', 'phase_p_adj', 'n_reaches', 'n_subjects', 'converged']])
recovery.to_parquet(CACHE_DIR / 'lmm_recovery.parquet', index=False)


## 5. -log10(adjusted p) summary

Three stacked bar charts -- one per analysis -- of the top features by
significance. Red dashed line marks the FDR cutoff.


In [ ]:
combined = pd.concat([
    omnibus.head(TOP_N_FIGURE).assign(analysis='Omnibus (all phases)'),
    deficit.head(TOP_N_FIGURE).assign(analysis='Deficit (Baseline -> Post_Injury_2-4)'),
    recovery.head(TOP_N_FIGURE).assign(analysis='Recovery (Post_Injury_2-4 -> Post_Rehab_Test)'),
])
combined['neg_log_p'] = -np.log10(combined['phase_p_adj'].clip(lower=1e-30))

sig_threshold = -np.log10(FDR_ALPHA)
fig, axes = plt.subplots(3, 1, figsize=FIGSIZE_BARS)
for ax, (label, group) in zip(axes, combined.groupby('analysis', sort=False)):
    plot_df = group.dropna(subset=['neg_log_p']).sort_values('neg_log_p')
    colors = ['steelblue' if v >= sig_threshold else 'lightgray' for v in plot_df['neg_log_p']]
    ax.barh(plot_df['feature'], plot_df['neg_log_p'], color=colors)
    ax.axvline(sig_threshold, color='red', linestyle='--', linewidth=1, label=f'FDR q={FDR_ALPHA}')
    ax.set_xlabel('-log10(FDR-adjusted p)')
    ax.set_title(label)
    ax.legend(loc='lower right')
plt.tight_layout()
stamp_version(fig, label='05 LMM summary')
plt.savefig(EXAMPLE_OUTPUT_DIR / '05_lmm_summary.png', dpi=150, bbox_inches='tight')
plt.show()


<!-- INTERPRETATION_BLOCK -->
## How to read this notebook's output

<details>
<summary>What the per-feature LMM bar chart tells you (click to expand)</summary>

**Three stacked bar charts**: omnibus phase effect, deficit delta
(Baseline vs Post_Injury_2-4), recovery delta (Post_Injury_2-4 vs
Post_Rehab_Test). Each plots `-log10(FDR-adjusted p)` per feature.

- Bars to the right of the red dashed line (`-log10(0.05)`) = features
  significantly affected by phase after multiple-testing correction.
- A long blue (significant) tail = many kinematic features change
  across phases; reaching kinematics is broadly affected.
- Few or no bars past the red line = no detectable phase effects after
  FDR; either the cohort is too small to find them or the effects are
  small relative to within-subject variability.

**At small N expect few features to clear FDR.** The LMM uses
reach-level data (hundreds of reaches per subject) which gives within-
subject precision, but the inferential N is still the 4 subjects.
The chi-square Wald approximation at small N means raw
p-values are slightly too small; FDR correction partially compensates.

**Per-analysis interpretation**:

- *Omnibus*: "is anything different across the four phases at all?".
  This is the broadest test; surviving features here are robustly
  phase-dependent.
- *Deficit*: "which features dropped between baseline and Post_Injury_2-4?".
  Surviving features = injury-affected aspects of reaching.
- *Recovery*: "which features came back between Post_Injury_2-4 and
  Post_Rehab_Test?". Surviving features = rehab-responsive aspects.

A feature surviving all three FDR cutoffs is a strong candidate for the
paper's "this is what reaching looks like at each phase" narrative.

</details>
